> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验十一：Qwen2.5五基础版算子统一接入


建议学时：1学时


# 实验任务


## 任务描述


本实验将五类自定义算子接入本地Qwen2.5-0.5B，验证替换后模型仍可完成前向计算，且输出结果与原生模型保持一致。


## 学习目标


了解在不改变模型权重的前提下替换内部计算模块的方法；理解动态库加载、跨设备数据转换、注意力层参数兼容和模型级正确性验证。


# 任务准备


## 实验环境准备


统一接入不是重新训练模型，而是在保持模型权重、输入处理和输出投影不变的条件下，替换其中的关键计算。五类算子分别承担归一化、位置编码、前馈激活、线性变换和注意力聚合等工作。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">模型</td>
<td style="text-align:left;">Models/Qwen2.5-0.5B</td>
</tr>
<tr>
<td style="text-align:left;">设备</td>
<td style="text-align:left;">宿主Ascend NPU</td>
</tr>
<tr>
<td style="text-align:left;">算子库</td>
<td style="text-align:left;">五个*Experiment/out/lib下的kernel/register .so</td>
</tr>
<tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">未padding的causal prompt；use_cache=False</td>
</tr>
<tr>
<td style="text-align:left;">计时</td>
<td style="text-align:left;">原生与custom各warmup 1次后repeat次前向均值</td>
</tr>
</tbody></table>


## 从自定义算子开发到Qwen2.5统一接入的技术路线


GEMM、RMSNorm、SwiGLU、RoPE和GQA Attention五个基础版算子的开发、单算子Golden验证、独立计时和Profiling已在前序单算子实验中完成。本实验把这些结果作为前置条件，不再重复内核实现，重点说明如何将已经可独立调用的动态库接入Qwen2.5：整理接入产物，建立原生基线，加载并注册算子，定位替换点，绑定自定义前向，处理数据边界，最后完成模型级验证与统一计时。


本实验的实际技术链路为：生成接入产物 → 建立原生基线 → 加载并注册算子 → 定位模型替换点 → 绑定自定义前向 → 适配数据边界 → 模型级验证 → 统一计时与分析。


基础版接入时，RoPE对Q和K分别调用基础算子；RMSNorm、RoPE和GQA Attention保持封装层的主机侧调用约定。GEMM与SwiGLU在边界将所需数据传入NPU，计算后再将结果交回原模型后续模块。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">环节</th>
<th style="text-align:left;">本实验中的具体操作</th>
<th style="text-align:left;">完成标志</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">前置条件</td>
<td style="text-align:left;">五个基础版算子已在各自实验中完成开发、Golden验证、独立计时和动态库构建；本实验不再重复单算子实现。</td>
<td style="text-align:left;">五类算子均能通过torch.ops入口独立调用。</td>
</tr>
<tr>
<td style="text-align:left;">1. 生成接入产物</td>
<td style="text-align:left;">汇总五类算子的设备内核库、PyTorch注册库、命名空间、函数签名和运行时依赖，形成统一接入清单。</td>
<td style="text-align:left;">动态库路径和接口清单完整，依赖可解析。</td>
</tr>
<tr>
<td style="text-align:left;">2. 建立原生基线</td>
<td style="text-align:left;">固定模型、分词器、提示词、缓存选项和精度，先运行未替换的Qwen2.5并保存logits与前向时间。</td>
<td style="text-align:left;">获得后续比较使用的Golden输出和原生时间。</td>
</tr>
<tr>
<td style="text-align:left;">3. 加载并注册算子</td>
<td style="text-align:left;">按设备内核库在前、框架注册库在后的顺序加载动态库，将五类算子注册到torch.ops。</td>
<td style="text-align:left;">五类命名空间和调用入口均可访问。</td>
</tr>
<tr>
<td style="text-align:left;">4. 定位模型替换点</td>
<td style="text-align:left;">遍历模型模块，将线性层、RMSNorm、MLP激活、RoPE和GQA Attention映射到五类自定义算子。</td>
<td style="text-align:left;">形成模块替换清单，并记录各类模块数量。</td>
</tr>
<tr>
<td style="text-align:left;">5. 绑定自定义前向</td>
<td style="text-align:left;">通过方法绑定或Monkey Patch替换目标模块的forward，只改写目标计算，保留权重及外围模型逻辑。</td>
<td style="text-align:left;">替换函数成功绑定，模型结构和参数保持不变。</td>
</tr>
<tr>
<td style="text-align:left;">6. 适配数据边界</td>
<td style="text-align:left;">在算子调用前后处理contiguous布局、展平与恢复、float32计算、CPU/NPU传输、偏置补回和注意力头映射。</td>
<td style="text-align:left;">每个替换模块的输入输出形状、设备和数据类型符合原模型约定。</td>
</tr>
<tr>
<td style="text-align:left;">7. 模型级验证</td>
<td style="text-align:left;">使用与原生基线相同的输入运行替换后模型，比较logits的最大误差、平均误差和allclose。</td>
<td style="text-align:left;">完整前向成功，误差满足统一阈值。</td>
</tr>
<tr>
<td style="text-align:left;">8. 统一计时与分析</td>
<td style="text-align:left;">两条路径采用相同预热和重复次数，保存每轮时间、平均时间、替换数量和误差结果。</td>
<td style="text-align:left;">得到可复现的模型级正确性与性能记录。</td>
</tr>
</tbody></table>


基础版与优化版遵循完全相同的模型接入顺序，差异只来自被加载的算子实现。基础版优先保证公式与代码易于核验，RoPE对Q和K分别调用，适合作为统一接入的功能基线。


# 任务实施


任务实施对应上表八个接入环节。前六步完成“把算子放进模型并保证接口兼容”，第七步验证替换后模型语义，第八步在统一口径下记录模型级时间。


## 步骤一：生成并核对统一接入产物


将五个基础版算子工程的构建结果整理为接入清单。每一项至少记录设备内核库、PyTorch注册库、torch.ops命名空间、函数签名、输入输出约束和依赖路径。统一脚本启动前逐项检查动态库文件是否存在，避免在模型已经加载后才发现某一算子缺失。基础版与优化版使用相同接口时，必须在不同Python进程中运行，防止命名空间重复注册。


本步完成标志：五类设备库和注册库均已生成，路径可解析，接口清单与统一接入脚本中的配置一致。


## 步骤二：运行原生Qwen2.5并建立Golden基线


在加载任何自定义动态库之前，先加载Qwen2.5模型与分词器，固定提示词、随机种子、精度和use_cache设置。原生路径先预热，再按统一重复次数运行，保存native_logits、每轮前向时间和平均时间。后续自定义路径必须复用同一模型权重和同一输入，原生logits作为模型级Golden结果。


本步完成标志：原生前向无报错，Golden logits和原生时间已保存，模型、输入与测试参数可复现。


## 步骤三：加载动态库并注册五类算子


统一脚本读取步骤一的清单，按依赖关系加载五类基础版设备内核库和注册库。设备内核库提供NPU可执行代码，注册库把C++封装暴露到torch.ops；二者都成功加载后，再逐项检查GEMM、RMSNorm、SwiGLU、RoPE和GQA Attention的调用入口是否存在。


```python
def load_library(torch, path):
```


```text
if not path.is_file():
```


```text
raise FileNotFoundError(path)
```


```python
torch.ops.load_library(str(path))
```


```text
# 每次基础/优化实验都在独立Python进程运行，避免相同namespace重复注册。
```


## 步骤四：遍历Qwen2.5并定位模型替换点


遍历Qwen2.5的模块树，将nn.Linear映射到GEMM，将RMSNorm层映射到RMSNorm，将MLP门控激活映射到SwiGLU，并在Attention内部定位RoPE和GQA Attention。定位阶段只建立模块到算子的映射，不立即修改计算；同时记录每类模块数量，供替换后核对。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">Qwen模块</th>
<th style="text-align:left;">替换算子</th>
<th style="text-align:left;">接入方法</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">Linear</td>
<td style="text-align:left;">gemm_custom.gemm</td>
<td style="text-align:left;">169个torch.nn.Linear；输入/权重转NPU，输出回CPU并加bias</td>
</tr>
<tr>
<td style="text-align:left;">RMSNorm</td>
<td style="text-align:left;">rmsnorm_ custom.rms_norm</td>
<td style="text-align:left;">替换49个Norm，保持eps与weight</td>
</tr>
<tr>
<td style="text-align:left;">MLP</td>
<td style="text-align:left;">swiglu_custom.swiglu</td>
<td style="text-align:left;">替换24个MLP激活gate/up后调用自定义算子</td>
</tr>
<tr>
<td style="text-align:left;">Attention/RoPE/GQA</td>
<td style="text-align:left;">RoPE + gqa_attention</td>
<td style="text-align:left;">替换24个attention；Q/K旋转后用GQA完成causal score/value聚合</td>
</tr>
</tbody></table>


替换边界必须明确：线性层保留原权重并在需要时补回偏置；RMSNorm保留weight和eps；MLP只替换门控激活；Attention保留Q/K/V投影、位置参数、掩码、缓存处理和输出投影，只替换Q/K位置旋转与注意力聚合。


## 步骤五：绑定自定义forward并完成模块替换


调用统一脚本中的patch_linear、patch_rmsnorm、patch_swiglu和patch_attention，通过Python方法绑定机制把已定位模块的forward替换为自定义实现。绑定时不复制或改写模型权重；补丁函数返回实际替换数量，并与步骤四的定位结果核对，防止漏替换或重复替换。


Attention补丁在原有投影完成后取得Q、K、V，再调用自定义RoPE和GQA Attention；基础版对Q、K分别调用RoPE，优化版使用Q/K紧凑合并调用，其余外围逻辑保持一致。


```text
q = self.q_proj(hidden_states).view(batch, sequence, q_heads, head_dim).transpose(1, 2)
```


```text
k = self.k_proj(hidden_states).view(batch, sequence, kv_heads, head_dim).transpose(1, 2)
```


```text
# baseline: 分别调用rope_baseline(q) 和rope_baseline(k)
```


```text
# optimized: 一次rope_qk_compact(q, k, cos, sin, sequence, q_heads, kv_heads)
```


```text
result = ops['gqa'](q.float().contiguous(), k.float().contiguous(),
```


```text
v.float().contiguous(), 0.0, True)
```


## 步骤六：适配张量形状、精度与设备边界


自定义forward调用算子前，将输入整理为算子要求的连续布局和float32计算格式。GEMM将高维输入展平为二维，返回后恢复原形状并补回偏置；RMSNorm传入原权重和eps；RoPE核对batch、序列长度、头数和head_dim；GQA Attention核对Q头与KV头的分组关系。当前封装需要跨设备时，在调用边界显式完成CPU到NPU及NPU到CPU转换，并将结果恢复为原模型期望的数据类型和形状。


本步完成标志：五类替换模块的输入输出形状、dtype和device与原模型约定一致，完整模型能够越过所有替换点执行到输出层。


## 步骤七：运行模型级Golden正确性验证


设置CANN环境后启动基础版统一接入脚本。脚本使用步骤二保存的相同输入，先确认五类动态库全部加载，再执行模块定位、forward绑定和替换后前向。测试应保存custom_logits、实际替换数量、最大绝对误差、平均绝对误差和allclose判定；若失败，可按GEMM、RMSNorm、SwiGLU、RoPE、GQA Attention的顺序逐类停用补丁，定位首个产生偏差的边界。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
python3 Qwen2.5BaselineIntegrationExperiment/qwen2_5_five_ops_benchmark.py \
```


```text
--model /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Models/Qwen2.5-0.5B --repeat 1
```


## 模型级正确性判定与验收标准


以原生logits为Golden结果，比较替换后logits的最大误差、平均误差和allclose。两条路径必须使用同一模型权重、同一token序列和同一缓存设置；只有完整前向成功、替换数量符合预期且allclose满足既定atol/rtol阈值，才判定模型级接入通过。


## 步骤八：统一计时、保存结果并分析接入开销


原生路径和自定义路径均先预热，再使用相同repeat次数统计每轮前向时间和平均时间；结果文件同时保存模型路径、提示词长度、缓存设置、替换数量、误差与判定结果。这里记录的是包含完整模型前向及当前CPU/NPU数据转换的接入时间，只能用于评价统一接入链路，不能替代前序单算子实验中的纯设备侧内核计时。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">指标</th>
<th style="text-align:left;">实测值</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">原生前向</td>
<td style="text-align:left;">280.864 ms</td>
</tr>
<tr>
<td style="text-align:left;">五算子自定义前向</td>
<td style="text-align:left;">50792.863 ms</td>
</tr>
<tr>
<td style="text-align:left;">native_over_custom</td>
<td style="text-align:left;">0.00553</td>
</tr>
<tr>
<td style="text-align:left;">max_abs_diff</td>
<td style="text-align:left;">0.001199722</td>
</tr>
<tr>
<td style="text-align:left;">allclose(atol=rtol=1e-2)</td>
<td style="text-align:left;">true</td>
</tr>
<tr>
<td style="text-align:left;">替换数量</td>
<td style="text-align:left;">GEMM 169；RMSNorm 49；SwiGLU 24；Attention/RoPE/GQA 24</td>
</tr>
</tbody></table>


结果文件保存在对应工程的结果目录中。本次以一次重复运行确认实机链路和正确性；如需得到稳定的性能统计，应固定测试环境并进行多轮采样。


# 任务拓展


可分别禁用一种算子替换，或改变提示词长度，观察各算子对模型级耗时和误差的影响；还可以将全部封装层改为常驻NPU的数据流，以减少跨设备拷贝。


# 测试与验收


## 五算子接入Qwen2.5 的前向传播测试


该命令先运行原生Qwen2.5 前向，再加载五个基础版动态库并替换相应模块，最后比较两条路径的输出。输出中的allclose=True表示模型级结果通过；记录的前向时间用于观察当前接入链路，不等于纯内核性能。基础版和优化版必须在不同Python进程中运行。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops
source /home/developer/Ascend/cann-8.5.2/set_env.sh
python3 Qwen2.5BaselineIntegrationExperiment/qwen2_5_five_ops_benchmark.py --model /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Models/Qwen2.5-0.5B --repeat 3


五个基础版算子已在同一条模型前向链路中完成加载、替换和输出校验。当前性能数值反映统一接入的调用边界；后续应重点减少跨设备往返、临时数据整理和重复传输。


## 附录：逐算子接入实现要点


线性层替换会遍历模型中的线性模块，将输入整理为二维数据后调用自定义矩阵乘法，并在返回结果后补回偏置。归一化层保留原有权重和稳定项；前馈层只替换中间激活；注意力层保留投影、位置编码、掩码、缓存和输出投影，只替换位置旋转和注意力聚合。


```python
def custom_linear(self, x):
```


```text
shape = x.shape[:-1]
```


```text
y = gemm(cpu_to_npu(x.reshape(-1, x.shape[-1]).float()),
```


```text
cpu_to_npu(self.weight.t().float().contiguous()))
```


```text
y = npu_to_cpu(y)
```


```text
if self.bias is not None: y = y + self.bias.float()
```


```text
return y.reshape(*shape, self.weight.shape[0]).to(dtype=x.dtype)
```


```python
def custom_rmsnorm(self, hidden_states):
```


```text
eps = getattr(self, 'variance_epsilon', getattr(self, 'eps', 1e-6))
```


```text
return rms(hidden_states.float().contiguous(), self.weight.float().contiguous(), float(eps))
```


## 前向时间与正确性保证


对同一模型和同一输入，先执行原生路径，再执行替换后的路径；两条路径均先预热。每次前向完成后记录耗时，最后计算输出结果的绝对误差，并保存每轮时间、替换数量和判定结果，确保过程可追溯。


```python
def timed_forward(torch, model, input_ids, repeat):
```


```text
model(input_ids=input_ids, use_cache=False).logits  # warmup
```


```text
values = []
```


```text
for _ in range(repeat):
```


```text
start = time.perf_counter()
```


```text
logits = model(input_ids=input_ids, use_cache=False).logits
```


```text
values.append((time.perf_counter() - start) * 1000.0)
```


```text
return logits, sum(values) / len(values), values
```


```text
diff = (custom_logits - native_logits).abs()
```


```python
ok = torch.allclose(native_logits, custom_logits, atol=1e-2, rtol=1e-2)
```


## 复现检查


运行前应确认十个工程的动态库均已生成，基础版和优化版需在不同的Python进程中运行。若输出不一致，应依次检查输入是否补齐、缓存是否关闭、位置编码的形状以及注意力头之间的映射关系。


## 完整接入链路说明


统一接入不改写Qwen的权重和配置。脚本先运行原生模型，再加载五类动态库并遍历模块，随后绑定替换后的前向方法。不同算子根据当前实现路径在主机或设备上执行，但都保持原模型的输入输出语义。


```text
ops = load_ops(torch, variant)
```


```text
patched = {
```


```text
'gemm_linear': patch_linear(torch, model, ops['gemm']),
```


```text
'rmsnorm': patch_rmsnorm(model, ops['rms']),
```


```text
'swiglu_mlp': patch_swiglu(model, ops['swiglu']),
```


```text
'attention_rope_gqa': patch_attention(model, ops, variant),
```


```text
}
```


```text
# 本次实测：169 / 49 / 24 / 24 个模块被替换
```


# 实验总结


本实验以五个已完成独立验证的基础版算子为前置条件，重点完成统一接入的八个环节：生成接入产物、建立原生基线、加载并注册算子、定位模型替换点、绑定自定义前向、适配数据边界、执行模型级验证以及统一计时与分析。整个过程不重新训练模型、不修改权重，通过同一输入下的原生logits与自定义logits对比确认接入语义，并将模型级接入时间与单算子纯内核性能明确区分。
